# Integración de nacimientos y mortalidad infantil

Exploración de `surviving_infants.parquet`, con nacimientos (`total_nacimientos`), defunciones de hombres y mujeres menores de un año (`total_defunciones_menor_1`) y el saldo del **año inmediatamente anterior**:

```text
sobrevivientes_edad_1(t) = total_nacimientos(t−1) − total_defunciones_menor_1(t−1)
```

Se excluyen los totales departamentales. La unión completa conserva las filas sin correspondencia y las registra en `unmatched_rows.csv`; `quality_report.json` contiene los conteos de validación. Los países con departamento nulo no se emparejan ni se agregan.

Para regenerar los tres archivos desde la raíz del repositorio:
```bash
.venv/bin/python scripts/processing/integrate_surviving_infants.py
```

Cuando existe births con clave completa pero falta la fila de mortality, las defunciones se asumen cero y no se reporta esa fila como sin correspondencia. Los conteos nulos de una fila existente no se imputan. El primer año no tiene saldo por falta del año previo. Los valores negativos se conservan para revisión. Este saldo de conteos anuales no es seguimiento individual de una cohorte.

El informe cuenta por separado las filas de mortality asumidas cero. Esas filas permanecen en el Parquet y participan en la resta del año siguiente, pero no aparecen en el CSV de filas sin correspondencia.

In [23]:
from pathlib import Path
import json

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import HTML, display

cwd = Path.cwd().resolve()
root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "scripts/processing/integrate_surviving_infants.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Ejecute el notebook dentro del repositorio data-processing")

data_dir = root / "data/processed/births_mortality_dane/integration"
parquet_path = data_dir / "surviving_infants.parquet"
unmatched_path = data_dir / "unmatched_rows.csv"
report_path = data_dir / "quality_report.json"
for path in (parquet_path, unmatched_path, report_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"No existe {path}. Ejecute scripts/processing/integrate_surviving_infants.py"
        )

surviving_infants = pd.read_parquet(parquet_path)
unmatched = pd.read_csv(
    unmatched_path,
    dtype={
        "ano": "Int64", "codigo_departamento": "string",
        "nombre_departamento": "string", "codigo_municipio": "string",
        "nombre_municipio": "string", "total_nacimientos": "Int64", "total_defunciones_menor_1": "Int64",
        "origen": "string", "motivo": "string",
    },
)
report = json.loads(report_path.read_text(encoding="utf-8"))
pd.set_option("display.max_columns", None)
print(f"Archivo: {parquet_path}")
print(f"Filas: {len(surviving_infants):,} | Columnas: {len(surviving_infants.columns)}")
print(f"Periodo: {surviving_infants['ano'].min()}–{surviving_infants['ano'].max()}")

Archivo: /home/diego/repos/data-processing/data/processed/births_mortality_dane/integration/surviving_infants.parquet
Filas: 21,868 | Columnas: 8
Periodo: 2008–2025


## Estructura, schema y nulos

In [24]:
print(pq.read_schema(parquet_path).remove_metadata())
schema = pd.DataFrame({
    "columna": surviving_infants.columns,
    "tipo_arrow": [str(field.type) for field in pq.read_schema(parquet_path)],
    "tipo_pandas": surviving_infants.dtypes.astype(str).to_numpy(),
    "nulos": surviving_infants.isna().sum().to_numpy(),
    "porcentaje_nulo": (100 * surviving_infants.isna().mean()).round(2).to_numpy(),
})
display(schema)

ano: int64
codigo_departamento: large_string
nombre_departamento: large_string
codigo_municipio: large_string
nombre_municipio: large_string
total_nacimientos: int64
total_defunciones_menor_1: int64
sobrevivientes_edad_1: int64


,columna,tipo_arrow,tipo_pandas,nulos,porcentaje_nulo
0,ano,int64,Int64,0,0.00
1,codigo_departamento,large_string,string,1092,4.99
2,nombre_departamento,large_string,string,0,0.00
3,codigo_municipio,large_string,string,0,0.00
4,nombre_municipio,large_string,string,0,0.00
5,total_nacimientos,int64,Int64,684,3.13
6,total_defunciones_menor_1,int64,Int64,408,1.87
7,sobrevivientes_edad_1,int64,Int64,2248,10.28


## Primeras 10 filas con todos los encabezados

In [25]:
display(surviving_infants.head(10))

,ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,sobrevivientes_edad_1
0,2008,05,ANTIOQUIA,-2,SIN INFORMACIÓN,483,8,<NA>
1,2008,05,ANTIOQUIA,05001,MEDELLÍN,33228,381,<NA>
2,2008,05,ANTIOQUIA,05002,ABEJORRAL,309,5,<NA>
3,2008,05,ANTIOQUIA,05004,ABRIAQUÍ,27,1,<NA>
4,2008,05,ANTIOQUIA,05021,ALEJANDRÍA,69,1,<NA>
5,2008,05,ANTIOQUIA,05030,AMAGÁ,361,1,<NA>
6,2008,05,ANTIOQUIA,05031,AMALFI,411,7,<NA>
7,2008,05,ANTIOQUIA,05034,ANDES,650,5,<NA>
8,2008,05,ANTIOQUIA,05036,ÁNGELÓPOLIS,85,0,<NA>
9,2008,05,ANTIOQUIA,05038,ANGOSTURA,179,5,<NA>


## Métricas principales

Los municipios identificados se cuentan por código departamental y código municipal de cinco dígitos. Los códigos `-1` y `-2` representan categorías especiales. El porcentaje de join real de cada entrada (sin incluir mortality asumida cero) usa como denominador sus filas después de excluir totales e incluye las claves incompletas.

In [26]:
matched = report["filas_con_join"]
valid_municipal_code = surviving_infants["codigo_municipio"].str.fullmatch(r"\d{5}", na=False)
identified = surviving_infants.loc[
    valid_municipal_code & surviving_infants["codigo_departamento"].notna()
]
metrics = pd.DataFrame([
    ("Filas consolidadas", len(surviving_infants)),
    ("Columnas", len(surviving_infants.columns)),
    ("Años", surviving_infants["ano"].nunique()),
    ("Códigos municipales identificados distintos", len(identified[["codigo_departamento", "codigo_municipio"]].drop_duplicates())),
    ("Claves con correspondencia real", matched),
    ("Filas de mortality asumidas cero", report["filas_mortality_asumidas_cero"]),
    ("Filas sin join de births", report["filas_sin_join"]["births"]["total"]),
    ("Filas sin join de mortality", report["filas_sin_join"]["mortality"]["total"]),
    ("Filas con sobrevivientes calculados", surviving_infants["sobrevivientes_edad_1"].notna().sum()),
    ("Sobrevivientes nulos", surviving_infants["sobrevivientes_edad_1"].isna().sum()),
    ("Sobrevivientes negativos", surviving_infants["sobrevivientes_edad_1"].lt(0).sum()),
], columns=["metrica", "valor"])
display(metrics)

coverage = pd.DataFrame([
    {
        "entrada": origin,
        "filas_sin_totales": report["filas_sin_totales"][origin],
        "filas_con_join": matched,
        "filas_mortality_asumidas_cero": report["filas_mortality_asumidas_cero"] if origin == "births" else 0,
        "filas_sin_join": report["filas_sin_join"][origin]["total"],
        "porcentaje_con_join": round(100 * matched / report["filas_sin_totales"][origin], 2),
    }
    for origin in ("births", "mortality")
])
display(coverage)

,metrica,valor
0,Filas consolidadas,21868
1,Columnas,8
2,Años,18
3,Códigos municipales identificados distintos,1123
4,Claves con correspondencia real,20322
5,Filas de mortality asumidas cero,454
6,Filas sin join de births,408
7,Filas sin join de mortality,684
8,Filas con sobrevivientes calculados,19620
9,Sobrevivientes nulos,2248


,entrada,filas_sin_totales,filas_con_join,filas_mortality_asumidas_cero,filas_sin_join,porcentaje_con_join
0,births,21184,20322,454,408,95.93
1,mortality,21006,20322,0,684,96.74


## Resumen por año

Las sumas siguientes corresponden a **las filas disponibles del consolidado**, con sus faltantes. No son totales nacionales: se excluyeron los totales de las entradas y hay cobertura territorial incompleta. La suma de sobrevivientes de un año usa insumos del año previo y solo filas donde pudo calcularse el saldo.

In [27]:
annual = surviving_infants.groupby("ano").agg(
    filas=("ano", "size"),
    filas_con_births=("total_nacimientos", "count"),
    filas_con_mortality=("total_defunciones_menor_1", "count"),
    nacimientos_disponibles=("total_nacimientos", lambda values: values.sum(min_count=1)),
    defunciones_menores_1_disponibles=("total_defunciones_menor_1", lambda values: values.sum(min_count=1)),
    filas_con_sobrevivientes=("sobrevivientes_edad_1", "count"),
    suma_sobrevivientes_disponibles=("sobrevivientes_edad_1", lambda values: values.sum(min_count=1)),
    sobrevivientes_nulos=("sobrevivientes_edad_1", lambda values: values.isna().sum()),
    sobrevivientes_negativos=("sobrevivientes_edad_1", lambda values: values.lt(0).sum()),
)
display(annual)

,filas,filas_con_births,filas_con_mortality,nacimientos_disponibles,defunciones_menores_1_disponibles,filas_con_sobrevivientes,suma_sobrevivientes_disponibles,sobrevivientes_nulos,sobrevivientes_negativos
ano,,,,,,,,,
2008,1191,1172,1173,714001,10490,0,<NA>,1191,0
2009,1203,1174,1183,698056,9509,1154,703252,49,0
2010,1210,1178,1186,653498,8291,1154,688297,56,1
2011,1208,1172,1190,665041,8121,1154,644973,54,1
2012,1209,1177,1186,676592,8177,1154,656715,55,0
2013,1213,1173,1194,658552,7596,1154,668168,59,2
2014,1206,1174,1186,669026,7563,1154,650774,52,4
2015,1217,1172,1199,660867,7221,1154,661262,63,2
2016,1223,1182,1195,647420,7212,1154,653496,69,3


## Ejemplos con sobrevivientes calculados

In [28]:
display(surviving_infants.loc[
    surviving_infants["sobrevivientes_edad_1"].notna()
].head(10))

,ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,sobrevivientes_edad_1
1191,2009,05,Antioquia,-2,SIN INFORMACIÓN,497,8,475
1192,2009,05,Antioquia,05001,MEDELLÍN,31893,333,32847
1193,2009,05,Antioquia,05002,ABEJORRAL,271,5,304
1194,2009,05,Antioquia,05004,ABRIAQUÍ,21,1,26
1195,2009,05,Antioquia,05021,ALEJANDRÍA,44,0,68
1196,2009,05,Antioquia,05030,AMAGÁ,354,2,360
1197,2009,05,Antioquia,05031,AMALFI,404,5,404
1198,2009,05,Antioquia,05034,ANDES,594,5,645
1199,2009,05,Antioquia,05036,ÁNGELÓPOLIS,73,0,85
1200,2009,05,Antioquia,05038,ANGOSTURA,160,2,174


In [29]:
# Cambiar este código para consultar otro municipio
codigo_municipio_consulta = "94663"

sobrevivientes_municipio = surviving_infants.loc[
    surviving_infants["codigo_municipio"].eq(codigo_municipio_consulta)
    & surviving_infants["sobrevivientes_edad_1"].notna()
].sort_values(["ano", "codigo_municipio"])

print(f"Municipio: {codigo_municipio_consulta}")
print(f"Filas con sobrevivientes calculados: {len(sobrevivientes_municipio):,}")
# La tabla contiene todas las filas del departamento y permite desplazarse.
display(HTML(
    '<div style="max-height:480px;overflow:auto">'
    + sobrevivientes_municipio.to_html(index=False, escape=True, na_rep="—")
    + '</div>'
))

Municipio: 94663
Filas con sobrevivientes calculados: 13


ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,sobrevivientes_edad_1
2009,94,Guainía,94663,MAPIRIPANA,4,0,2
2010,94,Guainía,94663,Mapiripana,3,0,4
2011,94,Guainía,94663,Mapiripana,1,0,3
2012,94,Guainía,94663,Mapiripana,0,0,1
2013,94,Guainía,94663,Mapiripana,0,0,0
2014,94,Guainía,94663,Mapiripana,0,0,0
2015,94,Guainía,94663,Mapiripana,3,0,0
2016,94,Guainía,94663,Mapiripana,3,0,3
2017,94,Guainía,94663,Mapiripana,4,0,3
2018,94,Guainía,94663,Mapiripana,15,0,4


## Consulta por municipio

In [30]:
# Cambiar este código para consultar otro municipio (conservar cinco dígitos).
codigo_municipio_consulta = "05004"

sobrevivientes_municipio = surviving_infants.loc[
    surviving_infants["codigo_municipio"].eq(codigo_municipio_consulta)
    & surviving_infants["sobrevivientes_edad_1"].notna()
].sort_values(["ano", "codigo_municipio"])

print(f"Municipio: {codigo_municipio_consulta}")
print(f"Filas con sobrevivientes calculados: {len(sobrevivientes_municipio):,}")
# La tabla contiene todas las filas seleccionadas del municipio y permite desplazarse.
display(HTML(
    '<div style="max-height:480px;overflow:auto">'
    + sobrevivientes_municipio.to_html(index=False, escape=True, na_rep="—")
    + '</div>'
))

Municipio: 05004
Filas con sobrevivientes calculados: 17


ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,sobrevivientes_edad_1
2009,05,Antioquia,05004,ABRIAQUÍ,21,1,26
2010,05,Antioquia,05004,Abriaquí,34,0,20
2011,05,Antioquia,05004,Abriaquí,30,1,34
2012,05,Antioquia,05004,Abriaquí,31,2,29
2013,05,Antioquia,05004,Abriaquí,22,0,29
2014,05,Antioquia,05004,Abriaquí,27,1,22
2015,05,Antioquia,05004,Abriaquí,24,0,26
2016,05,Antioquia,05004,Abriaquí,12,0,24
2017,05,Antioquia,05004,Abriaquí,16,0,12
2018,05,Antioquia,05004,Abriaquí,15,0,16


## Filas sin correspondencia: resumen

`sin_correspondencia` indica que la clave completa solo está en una entrada.
`clave_incompleta` indica que falta parte de la clave: en los archivos actuales son países extranjeros con departamento nulo. Estas filas se conservan separadas por origen para evitar una unión de muchos países entre sí.

In [31]:
unmatched_summary = (
    unmatched.groupby(["origen", "motivo"], dropna=False).size()
    .unstack(fill_value=0)
)
unmatched_summary["total_nacimientos"] = unmatched_summary.sum(axis=1)
display(unmatched_summary)

display(
    unmatched.groupby(["ano", "origen", "motivo"], dropna=False).size()
    .unstack(["origen", "motivo"], fill_value=0)
)

print("Filas de mortality con clave completa sin births:")
display(unmatched.loc[
    unmatched["origen"].eq("mortality") & unmatched["motivo"].eq("sin_correspondencia")
])

motivo,clave_incompleta,total_nacimientos
origen,,
births,408,408
mortality,684,684


origen,births,mortality
motivo,clave_incompleta,clave_incompleta
ano,,
2008,18,19
2009,20,29
2010,24,32
2011,18,36
2012,23,32
2013,19,40
2014,20,32
2015,18,45


Filas de mortality con clave completa sin births:


,ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,origen,motivo


## Detalle completo de filas sin correspondencia

La tabla muestra **todas** las filas del filtro en un panel con desplazamiento. Por defecto incluye el archivo completo. Cambiar los valores de la celda para filtrar por año, origen, motivo o municipio; usar `None` para quitar un filtro. Para filtrar un municipio identificado, conservar sus cinco dígitos, por ejemplo `"11001"`.

In [32]:
ano_filtro = None            # Ejemplo: 2019
origen_filtro = None         # "births" o "mortality"
motivo_filtro = None         # "sin_correspondencia" o "clave_incompleta"
codigo_municipio_filtro = None  # Ejemplo: "11001" o "-1"

unmatched_filtered = unmatched.copy()
for column, value in (
    ("ano", ano_filtro), ("origen", origen_filtro),
    ("motivo", motivo_filtro), ("codigo_municipio", codigo_municipio_filtro),
):
    if value is not None:
        unmatched_filtered = unmatched_filtered.loc[unmatched_filtered[column].eq(value)]
print(f"Filas mostradas: {len(unmatched_filtered):,} de {len(unmatched):,}")
display(HTML(
    '<div style="max-height:480px;overflow:auto">'
    + unmatched_filtered.to_html(index=False, escape=True, na_rep="—")
    + '</div>'
))

Filas mostradas: 1,092 de 1,092


ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,origen,motivo
2008,<NA>,EXTRANJEROS,-1,ALEMANIA,1,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,ARUBA,6,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,BRASIL,68,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,BÉLGICA,1,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,CHILE,1,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,CHINA,1,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,CROACIA,1,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,ECUADOR,14,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,ESTADOS UNIDOS DE AMÉRICA,9,<NA>,births,clave_incompleta
2008,<NA>,EXTRANJEROS,-1,GEORGIA DEL SUR E ISLAS SANDWICH DEL SUR,1,<NA>,births,clave_incompleta


## Revisión de negativos y nulos

Los negativos se conservan tal como resulta la resta y deben revisarse junto con el tipo de entidad. En 2008 todos los saldos son nulos por falta de 2007. Los demás nulos pueden deberse a nacimientos del año previo ausentes, medidas nulas en filas existentes o claves incompletas. La ausencia de una fila de mortality para una clave completa de births se asume cero.

In [33]:
negative_balances = surviving_infants.loc[
    surviving_infants["sobrevivientes_edad_1"].lt(0).fillna(False)
]
print(f"Saldos negativos: {len(negative_balances)}")
with pd.option_context("display.max_rows", None):
    display(negative_balances)

missing_balances = surviving_infants.loc[surviving_infants["sobrevivientes_edad_1"].isna()]
missing_summary = missing_balances.assign(
    tipo_fila=missing_balances["codigo_municipio"].map(
        lambda value: "extranjero" if value == "-1" else
        ("sin informacion municipal" if value == "-2" else "municipio identificado")
    )
).groupby(["ano", "tipo_fila"]).size().unstack(fill_value=0)
display(missing_summary)

Saldos negativos: 16


,ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total_nacimientos,total_defunciones_menor_1,sobrevivientes_edad_1
3536,2010,97,Vaupés,-2,Sin Información,4,0,-1
4453,2011,66,Risaralda,-2,Sin Información,3,1,-1
6605,2013,27,Chocó,-2,Sin Información,2,0,-1
7048,2013,76,Valle del Cauca,-2,Sin Información,0,0,-1
7701,2014,25,Cundinamarca,-2,Sin Información,1,0,-1
7934,2014,50,Meta,-2,Sin Información,0,0,-1
8098,2014,68,Santander,-2,Sin Información,6,0,-1
8383,2014,99,Vichada,-2,Sin Información,2,0,-1
8807,2015,19,Cauca,-2,Sin Información,0,0,-1
9235,2015,54,Norte de Santander,-2,Sin Información,0,0,-1


tipo_fila,extranjero,municipio identificado,sin informacion municipal
ano,,,
2008,37,1122,32
2009,49,0,0
2010,56,0,0
2011,54,0,0
2012,55,0,0
2013,59,0,0
2014,52,0,0
2015,63,0,0
2016,69,0,0


## Consistencia entre el Parquet y los reportes

Estas comprobaciones detectan que los archivos cargados mantengan los mismos conteos. Si alguna falla, regenerar los tres archivos con el script de integración antes de continuar.

In [12]:
assert len(surviving_infants) == report["filas_salida"]
assert len(unmatched) == sum(values["total"] for values in report["filas_sin_join"].values())
assert len(surviving_infants) == report["filas_con_join"] + report["filas_mortality_asumidas_cero"] + len(unmatched)
assert not surviving_infants["nombre_municipio"].str.strip().str.casefold().eq("total").any()
assert surviving_infants["sobrevivientes_edad_1"].isna().sum() == report["sobrevivientes_nulos"]
assert surviving_infants["sobrevivientes_edad_1"].lt(0).sum() == report["sobrevivientes_negativos"]
for origin in ("births", "mortality"):
    origin_rows = unmatched.loc[unmatched["origen"].eq(origin)]
    counts = report["filas_sin_join"][origin]
    assert len(origin_rows) == counts["total"]
    for reason in ("sin_correspondencia", "clave_incompleta"):
        assert origin_rows["motivo"].eq(reason).sum() == counts[reason]
print("Los conteos del Parquet, el CSV de filas sin join y el informe JSON coinciden.")

Los conteos del Parquet, el CSV de filas sin join y el informe JSON coinciden.
